В этом исследовании:
- Знакомимся с исходными CSV и изучаем полноту, уникальность, валидность и достоверность данных.
- На основе наблюдений определяем преобразования и проверки для STAGING и витрины. В raw будет загрузка as is 

### Смотрю общую структуру данных

In [2]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.appName("research").getOrCreate()

dfs = {}

for year in range(2020, 2024):
    df = (
        spark.read
        .option("header", True)
        .option("sep", ";")
        .csv(f"/home/jovyan/data/raw_{year}.csv")
    )

    dfs[year] = df
    print(f"\n{year}: {df.count()} строк, {len(df.columns)} столбцов")
    df.select(
        *[F.col(f"`{name}`") for name in df.columns[:10]]
    ).show(5, truncate=False)



2020: 289 строк, 348 столбцов
+--------------+--------------+-------+-------+-------+-------+-------+-------+-------+-------+
|Country/Region|Province/State|1/22/20|1/23/20|1/24/20|1/25/20|1/26/20|1/27/20|1/28/20|1/29/20|
+--------------+--------------+-------+-------+-------+-------+-------+-------+-------+-------+
|Afghanistan   |NULL          |0      |0      |0      |0      |0      |0      |0      |0      |
|Albania       |NULL          |0      |0      |0      |0      |0      |0      |0      |0      |
|Algeria       |NULL          |0      |0      |0      |0      |0      |0      |0      |0      |
|Andorra       |NULL          |0      |0      |0      |0      |0      |0      |0      |0      |
|Angola        |NULL          |0      |0      |0      |0      |0      |0      |0      |0      |
+--------------+--------------+-------+-------+-------+-------+-------+-------+-------+-------+
only showing top 5 rows


2021: 289 строк, 368 столбцов
+--------------+--------------+----------+-------


2022: 289 строк, 368 столбцов
+--------------+--------------+----------+----------+----------+----------+----------+----------+----------+----------+
|Country/Region|Province/State|01.01.2022|01.02.2022|01.03.2022|01.04.2022|01.05.2022|01.06.2022|01.07.2022|01.08.2022|
+--------------+--------------+----------+----------+----------+----------+----------+----------+----------+----------+
|Afghanistan   |NULL          |7356      |7361      |7364      |7365      |7367      |7367      |7368      |7369      |
|Albania       |NULL          |3217      |3220      |3220      |3224      |3224      |3228      |3233      |3233      |
|Algeria       |NULL          |6284      |6291      |6298      |6303      |6310      |6318      |6325      |6330      |
|Andorra       |NULL          |140       |140       |140       |141       |141       |141       |141       |141       |
|Angola        |NULL          |1772      |1772      |1775      |1778      |1789      |1799      |1809      |1819      |
+--------


2023: 289 строк, 71 столбцов
+--------------+--------------+----------+----------+----------+----------+----------+----------+----------+----------+
|Country/Region|Province/State|01.01.2023|01.02.2023|01.03.2023|01.04.2023|01.05.2023|01.06.2023|01.07.2023|01.08.2023|
+--------------+--------------+----------+----------+----------+----------+----------+----------+----------+----------+
|Afghanistan   |NULL          |7849      |7849      |7850      |7850      |7850      |7850      |7850      |7853      |
|Albania       |NULL          |3595      |3595      |3596      |3596      |3596      |3596      |3596      |3596      |
|Algeria       |NULL          |6881      |6881      |6881      |6881      |6881      |6881      |6881      |6881      |
|Andorra       |NULL          |165       |165       |165       |165       |165       |165       |165       |165       |
|Angola        |NULL          |1930      |1930      |1930      |1930      |1930      |1930      |1930      |1930      |
+---------

Провинции могут отсутствовать, а названия столбцов дат имеют разные форматы (через . и через /)

- Ниже проверю, статичен ли набор "страна + провинция" по годам
- И какие значения Country/Region представлены несколькими строками.


In [3]:
pairs_by_year = {} # словарь для пар (год: словарь страна+регион) 

for year, df in sorted(dfs.items()):
    pairs = {
        (row[0], row[1])
        for row in df.select(
            "Country/Region",
            F.trim(F.col("Province/State")),
        ).distinct().collect()
    } # считаем уникальные пары country/region +province/state

    pairs_by_year[year] = pairs 

    print(
        f"{year}: строк — {df.count()}, "
        f"уникальных пар — {len(pairs)}, "
        f"стран — {len({country for country, province in pairs})}"
    )

intersection = set.intersection(*pairs_by_year.values()) # разворачиваем значения словаря (другие словари) в структуру, делаем логическое пересечение 

print("\nПар в пересечении:", len(intersection))
print("Наборы совпадают:", all(pairs == intersection for pairs in pairs_by_year.values())) # возвращаем True, при полном совпадении каждой пары с общим пересечением
print("Число стран в пересечении:", len({country for country, province in intersection}))

# ------- далее считаем страны, представленные несколькими регионами 

print('------Ниже смотрим на предмет логических ошибок в странах, которые не отловить так просто-----------------')
regions = spark.createDataFrame(
    list(intersection),
    "country string, province string",
)

regions.groupBy("country").agg(
    F.count("*").alias("rows"),
    F.count(
        F.when(F.trim(F.col("province")) != "", F.col("province")) # считаем только строки, где не пустая провинция. Группировка сделает дедубликацию
    ).alias("regions"),
).orderBy(F.desc("regions"), "country").show(300, truncate=False)

2020: строк — 289, уникальных пар — 289, стран — 200
2021: строк — 289, уникальных пар — 289, стран — 200
2022: строк — 289, уникальных пар — 289, стран — 200
2023: строк — 289, уникальных пар — 289, стран — 200

Пар в пересечении: 289
Наборы совпадают: True
Число стран в пересечении: 200
------Ниже смотрим на предмет логических ошибок в странах, которые не отловить так просто-----------------
+--------------------------------+----+-------+
|country                         |rows|regions|
+--------------------------------+----+-------+
|China                           |34  |34     |
|Canada                          |16  |16     |
|United Kingdom                  |15  |14     |
|France                          |12  |11     |
|Australia                       |8   |8      |
|Netherlands                     |5   |4      |
|Denmark                         |3   |2      |
|Korea                           |2   |2      |
|New Zealand                     |3   |2      |
|Afghanistan               

+--------------------------------+----+-------+
|country                         |rows|regions|
+--------------------------------+----+-------+
|China                           |34  |34     |
|Canada                          |16  |16     |
|United Kingdom                  |15  |14     |
|France                          |12  |11     |
|Australia                       |8   |8      |
|Netherlands                     |5   |4      |
|Denmark                         |3   |2      |
|Korea                           |2   |2      |
|New Zealand                     |3   |2      |
|Afghanistan                     |1   |0      |
|Albania                         |1   |0      |
|Algeria                         |1   |0      |
|Andorra                         |1   |0      |
|Angola                          |1   |0      |
|Antarctica                      |1   |0      |
|Antigua and Barbuda             |1   |0      |
|Argentina                       |1   |0      |
|Armenia                         |1   |0

В исходных файлах по 289 уникальных пар "страна + провинция", пересечение содержит те же 289 пар.
Повторов пар нет. 200 значений Country/Region — это метки источника, а не 200 подтверждённых стран, потому что не все строки это страны 

Логические вопросы вызывают Олимпиады, MS Zaandam, Diamond Princess и Antarctica. Фильтрацию стран, и обработку
географии применяем при построении STAGING;

Holy See сохраняем, опустим вопросы отношений Святого Престола и Ватикана. 
Два Congo - норма, так и есть


## Проверка дат

Заголовки CSV — строки: среди них могут быть даты разных форматов и поля без даты.
Разделим их на три списка каждый год: даты через /, даты через . и остальные

Это нужно для разбора дат при построении STAGING; в RAW заголовки сохраняем как в источнике, я уже говорил 

In [4]:
from datetime import datetime

columns_by_year = {} # делается словарь, в котором будет 4 ключа, по одному на каждый год. Значением будем другой словарь с набором ключей по форматам дат /,.,и все остальные 
for year, df in sorted(dfs.items()): # ну собственно сортируемся по кортежам ранее сформированного словаря. Кортежи сравниваются как (1,2)>(2,2), (3,2)>(2,3) и т.д.
    groups = {"/": [], ".": [], "other": []} # словарь про который говорил выше
    for name in df.columns[2:]:  # Без столбцов страны и провинции.
        for group, fmt in [("/", "%m/%d/%y"), (".", "%m.%d.%Y")]: # проходимся по форматам,если удается преобразовать - падения не будет, в значение по текущему ключу добавим эту дату и остановим цикл перебора формата. 
            try:
                datetime.strptime(name, fmt)
            except ValueError: 
                continue # Иначе просто скипаем текущую итерацию, переходим к следующему формату 
            groups[group].append(name)
            break
        groups["other"].append(name)

    columns_by_year[year] = groups # не возвращается, но дальше по ноутбуку пригодится, чтобы мэтчить первые 2 столбца с безымянными с_
    print(f"\n{year}")
    for group, names in groups.items():
        print(f"{group} ({len(names)}): {names}") # ну собственно для каждого года выводим длину каждого значения для ключей в словаре-значении, и полный его список



2020
/ (213): ['1/22/20', '1/23/20', '1/24/20', '1/25/20', '1/26/20', '1/27/20', '1/28/20', '1/29/20', '1/30/20', '1/31/20', '2/13/20', '2/14/20', '2/15/20', '2/16/20', '2/17/20', '2/18/20', '2/19/20', '2/20/20', '2/21/20', '2/22/20', '2/23/20', '2/24/20', '2/25/20', '2/26/20', '2/27/20', '2/28/20', '2/29/20', '3/13/20', '3/14/20', '3/15/20', '3/16/20', '3/17/20', '3/18/20', '3/19/20', '3/20/20', '3/21/20', '3/22/20', '3/23/20', '3/24/20', '3/25/20', '3/26/20', '3/27/20', '3/28/20', '3/29/20', '3/30/20', '3/31/20', '4/13/20', '4/14/20', '4/15/20', '4/16/20', '4/17/20', '4/18/20', '4/19/20', '4/20/20', '4/21/20', '4/22/20', '4/23/20', '4/24/20', '4/25/20', '4/26/20', '4/27/20', '4/28/20', '4/29/20', '4/30/20', '5/13/20', '5/14/20', '5/15/20', '5/16/20', '5/17/20', '5/18/20', '5/19/20', '5/20/20', '5/21/20', '5/22/20', '5/23/20', '5/24/20', '5/25/20', '5/26/20', '5/27/20', '5/28/20', '5/29/20', '5/30/20', '5/31/20', '6/13/20', '6/14/20', '6/15/20', '6/16/20', '6/17/20', '6/18/20', '6/19

В каждом году есть оба формата дат и по одному столбцу без даты.
Форматы объединим при переходе RAW → STAGING в один столбец date,
а содержимое безымянных полей сейчас посмотрим в контексте локации. Подключим ИИ для рутинной задачи по подтягиванию локации для данных столбцов _с... в каждом году, и проверим его


In [8]:
result = None

for year, df in sorted(dfs.items()):
    groups = columns_by_year[year]
    unknown = sorted(set(groups["other"]) - set(groups["/"] + groups["."])) # смотрим на разницу множеств, чтобы получить безымянные столбцы. И только на их основе строить датафрейм для вывода 

    # Берём страну, провинцию и неизвестные столбцы; добавляем год к названиям
    current = df.select(
        "Country/Region",
        F.coalesce(F.col("Province/State"), F.lit("")).alias("Province/State"),
        *[F.col(f"`{c}`").alias(f"{year}_{c}") for c in unknown],
    )

    # Ставим значения разных лет рядом по одной локации
    result = current if result is None else result.join(
        current, ["Country/Region", "Province/State"], "full"
    )

# Показываем строки, где есть хотя бы одно неизвестное значение
result.dropna(how="all", subset=result.columns[2:]).show(truncate=False)

+--------------+--------------+----------+----------+----------+---------+
|Country/Region|Province/State|2020__c347|2021__c367|2022__c367|2023__c70|
+--------------+--------------+----------+----------+----------+---------+
|Korea         | North        |6         |6         |6         |6        |
|Korea         | South        |34093     |34093     |34093     |34093    |
+--------------+--------------+----------+----------+----------+---------+



In [6]:
# посмотрим, не наврали ли ИИ

dfs[2020].select("Country/Region", "Province/State", "_c347").show(300, truncate=False)

+--------------------------------+--------------------------------------------+-----+
|Country/Region                  |Province/State                              |_c347|
+--------------------------------+--------------------------------------------+-----+
|Afghanistan                     |NULL                                        |NULL |
|Albania                         |NULL                                        |NULL |
|Algeria                         |NULL                                        |NULL |
|Andorra                         |NULL                                        |NULL |
|Angola                          |NULL                                        |NULL |
|Antarctica                      |NULL                                        |NULL |
|Antigua and Barbuda             |NULL                                        |NULL |
|Argentina                       |NULL                                        |NULL |
|Armenia                         |NULL                

Да, похоже на правду. Ну собственно безымянное столбцы относятся к Кореям, у которых дубли между годами. За точность и достоверность таких данных не ручаюсь, принимаю решение не включать в витрину 

### Пропуски и простая статистика по исходным данным

При помощи ИИ сделаем сводку для получения агрегатов из дескрайбов каждого года. Задача - зафиксирировать для исходной схемы данных, имеют ли место быть null, не ожиданные типы для полей, и проверить файлы на общую адекватность, что смертей с каждым годом накапливается соразмерно, а не +100000 в другой год, и +100 в другой, при условии полных лет

In [3]:
values = None
for year, df in sorted(dfs.items()):
    names = columns_by_year[year]["/"] + columns_by_year[year]["."]
    current = df.select(
        F.col("Country/Region").alias("country"), F.lit(year).alias("year"),
        F.explode(F.array(*[F.col(f"`{name}`") for name in names])).alias("value"),
    )
    values = current if values is None else values.unionByName(current)

# Нечисловое значение не теряем среди NULL, считаем отдельно
values = (values.withColumn("missing", F.col("value").isNull() | (F.trim("value") == ""))
          .withColumn("number", F.expr("try_cast(value as bigint)"))
          .withColumn("invalid", ~F.col("missing") & F.col("number").isNull()))
values.groupBy("year").agg(
    F.count("*").alias("cells"), F.sum(F.col("missing").cast("int")).alias("nulls"),
    F.sum(F.col("invalid").cast("int")).alias("not_numeric"),
    F.min("number").alias("min"), F.max("number").alias("max"),
    F.round(F.avg("number"), 1).alias("mean"),
).orderBy("year").show(truncate=False)


+----+------+-----+-----------+---+-------+-------+
|year|cells |nulls|not_numeric|min|max    |mean   |
+----+------+-----+-----------+---+-------+-------+
|2020|99705 |0    |2          |0  |350604 |2381.7 |
|2021|105485|0    |0          |0  |825468 |13439.4|
|2022|105485|0    |0          |0  |1092764|21813.3|
|2023|19652 |0    |0          |0  |1123836|23595.0|
+----+------+-----+-----------+---+-------+-------+



In [4]:
# Для каждой страны ставим годы рядом: сначала те, где нашлись проблемы
by_country = values.groupBy("country", "year").agg(
    F.sum(F.col("missing").cast("int")).alias("nulls"),
    F.sum(F.col("invalid").cast("int")).alias("not_numeric"),
)
by_country.groupBy("country").pivot("year", sorted(dfs)).agg(
    F.first("nulls").alias("nulls"), F.first("not_numeric").alias("not_numeric"),
).orderBy(F.col("2020_not_numeric").desc(), *[
    F.col(f"{year}_nulls").desc() for year in sorted(dfs)
], "country").show(25, truncate=False) # limit можно увеличить, если нужны все страны


+----------------------+----------+----------------+----------+----------------+----------+----------------+----------+----------------+
|country               |2020_nulls|2020_not_numeric|2021_nulls|2021_not_numeric|2022_nulls|2022_not_numeric|2023_nulls|2023_not_numeric|
+----------------------+----------+----------------+----------+----------------+----------+----------------+----------+----------------+
|Korea                 |0         |2               |0         |0               |0         |0               |0         |0               |
|Afghanistan           |0         |0               |0         |0               |0         |0               |0         |0               |
|Albania               |0         |0               |0         |0               |0         |0               |0         |0               |
|Algeria               |0         |0               |0         |0               |0         |0               |0         |0               |
|Andorra               |0         |0     

Ожидаемо, 2 не числовых столбца с Кореями в 2020. Среднее растёт равномерно год-от-года, дополнительных преобразований при получении стейджа не надо

### Что делаем дальше в ETL

**STAGING:**
- Не берем 7 строк из исходных файлов, которые не страны
- Даты из заголовков через `/` и `.` (месяц первым) переносим в поле `date` типа DATE. В RAW заголовки не переименовываем.
- Проверяем числа, уникальность "страна + провинция + дата" и непрерывность дат. Пустая провинция допустима

**Витрина:**
- Учитываем, что страна может быть разбита на провинции: складываем их накопительные значения на одну дату.
- В исследовании данные начинаются 22 января 2020 и заканчиваются 9 марта 2023: ставим флажки на неполноту месяца и года